# Génération de `variables_modele.tex`

Ce notebook reconstruit le tableau LaTeX `variables_modele.tex` à partir de
`clean_dataset.csv`.

- Variables **continues** → `min / mean / max`
- Variables **catégorielles** → proportion de chaque modalité (mean d'un
  indicateur 0/1 par ligne)
- Variables **socio-demographic** (genre / age / expérience / distance) →
  agrégées au niveau **rider** (`rider_id`) pour ne pas pondérer par la durée
  des trajets


## 1. Imports & chargement

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

DATASET_PATH = "/Volumes/My Passport/NEWMOB/clean_dataset.csv"
OUTPUT_PATH  = "/Volumes/My Passport/NEWMOB/variables_modele_2.tex"

df = pd.read_csv(DATASET_PATH)
n_rows   = len(df)
n_trips  = df["source"].nunique() if "source" in df.columns else 0
n_riders = df["rider_id"].nunique() if "rider_id" in df.columns else 0
print(f"Loaded {n_rows:,} rows  |  trajets : {n_trips}  |  riders : {n_riders}")
print(f"Colonnes : {len(df.columns)}")


Loaded 396,483 rows  |  trajets : 31  |  riders : 16
Colonnes : 86


## 2. Pré-traitement (binning expérience)

In [3]:
# La colonne `experience` du Excel participants contient déjà des labels
# texte (<0.5, 0.5-1, 1-2, >2). Si jamais elle contenait des années numériques,
# on bascule sur un binning numérique compatible.
def bin_experience(x):
    if pd.isna(x):
        return np.nan
    try:
        x = float(x)
    except (TypeError, ValueError):
        return str(x).strip()  # déjà sous forme de label
    if x < 0.5: return "<0.5"
    if x < 1:   return "0.5-1"
    if x < 2:   return "1-2"
    return ">2"

if "experience" in df.columns:
    df["experience_bin"] = df["experience"].apply(bin_experience)
    print("experience_bin :", df["experience_bin"].value_counts(dropna=False).to_dict())


experience_bin : {'>2': 248924, '<0.5': 91830, '1-2': 46830, '0.5-1': 8899}


## 3. Helpers de stats

In [4]:
def _series(col, agg):
    """Renvoie la Series à analyser : df[col] ou un agrégat par rider."""
    if col not in df.columns:
        return None
    if agg == "per_rider" and "rider_id" in df.columns:
        return df.groupby("rider_id")[col].first()
    return df[col]


def stats_cont(col, agg=None):
    """min / mean / max formatés (1 décimale pour min/max, 2 pour mean)."""
    s = _series(col, agg)
    if s is None:
        return ("—", "—", "—")
    s = pd.to_numeric(s, errors="coerce").dropna()
    if s.empty:
        return ("—", "—", "—")
    return (f"{s.min():.1f}", f"{s.mean():.2f}", f"{s.max():.1f}")


def stats_modality(col, mod, agg=None):
    """Proportion d'une modalité dans une variable catégorielle."""
    s = _series(col, agg)
    if s is None:
        return ("—", "—", "—")
    s = s.dropna().astype(str).str.strip()
    if s.empty:
        return ("—", "—", "—")
    mask = s.str.lower() == str(mod).strip().lower()
    return (f"{int(mask.min())}.0", f"{mask.mean():.2f}", f"{int(mask.max())}.0")


## 4. Helpers de rendu LaTeX

In [5]:
def render_section_header(title):
    return [
        f"\\multicolumn{{6}}{{l}}{{\\textit{{\\small {title}}}}} \\\\",
        "\\addlinespace[1pt]",
    ]


def render_cont_row(col, label, source, agg=None):
    mn, mean, mx = stats_cont(col, agg)
    return [f"  {label} & \\textit{{cont.}} & {mn} & {mean} & {mx} & {source} \\\\"]


def render_cat_block(col, label, modalities, source, agg=None):
    n = len(modalities)
    lines = []
    for i, mod in enumerate(modalities):
        mn, mean, mx = stats_modality(col, mod, agg)
        if i == 0:
            lines.append(
                f"  \\multirow{{{n}}}{{3.8cm}}{{{label}}} & {mod} & {mn} & {mean} & {mx} & {source} \\\\"
            )
        else:
            lines.append(f"   & {mod} & {mn} & {mean} & {mx} &  \\\\")
    return lines


## 5. Configuration des variables

In [6]:
VIDEO  = "Video annotation"
GPSIMU = "GPS / IMU"
GPS    = "GPS"
GPSTS  = "GPS timestamp"
GIS    = "OSM / GIS"
QUEST  = "Questionnaire"
MODEL  = "Distance model"

SECTIONS = [
    ("Kinematics", [
        {"kind": "cont", "col": "speed_kmh", "label": "Speed (km/h)", "source": GPSIMU},
    ]),
    ("VRU — Presence", [
        {"kind": "cont", "col": "n_pedestrians", "label": "No. pedestrians in frame", "source": VIDEO},
        {"kind": "cont", "col": "n_cyclists",    "label": "No. cyclists in frame",    "source": VIDEO},
        {"kind": "cont", "col": "n_elderly",     "label": "No. elderly persons",       "source": VIDEO},
        {"kind": "cont", "col": "n_children",    "label": "No. children",              "source": VIDEO},
        {"kind": "cont", "col": "n_running",     "label": "No. pedestrians running",   "source": VIDEO},
    ]),
    ("VRU — Behaviour", [
        {"kind": "cont", "col": "n_pedestrians_crossing",       "label": "No. pedestrians crossing",              "source": VIDEO},
        {"kind": "cont", "col": "n_pedestrians_opposite",       "label": "No. pedestrians in opposite direction", "source": VIDEO},
        {"kind": "cont", "col": "n_pedestrians_same_direction", "label": "No. pedestrians in same direction",     "source": VIDEO},
        {"kind": "cont", "col": "n_pedestrians_stationary",     "label": "No. pedestrians stationary",            "source": VIDEO},
        {"kind": "cont", "col": "n_cyclists_crossing",          "label": "No. cyclists crossing",                  "source": VIDEO},
    ]),
    ("VRU — Trip proportion", [
        {"kind": "cont", "col": "prop_interaction_same_direction",     "label": "Share of same-direction interactions",     "source": VIDEO},
        {"kind": "cont", "col": "prop_interaction_opposite_direction", "label": "Share of opposite-direction interactions", "source": VIDEO},
        {"kind": "cont", "col": "prop_interaction_crossing",           "label": "Share of crossing interactions",            "source": VIDEO},
        {"kind": "cont", "col": "prop_interaction_stationary",         "label": "Share of stationary interactions",          "source": VIDEO},
    ]),
    ("VRU — Distance (predicted)", [
        {"kind": "cont", "col": "min_distance_m",  "label": "Distance to nearest VRU (m)",  "source": MODEL},
        {"kind": "cont", "col": "mean_distance_m", "label": "Mean VRU distance (m)",         "source": MODEL},
        {"kind": "cont", "col": "max_distance_m",  "label": "Distance to farthest VRU (m)", "source": MODEL},
    ]),
    ("Environment", [
        {"kind": "cat", "col": "WEATHER_LABEL",           "label": "Weather conditions",
         "modalities": ["No adverse", "Adverse"],                "source": VIDEO},
        {"kind": "cat", "col": "LIGHTING_LABEL",          "label": "Lighting conditions",
         "modalities": ["Daylight", "Dawn/Dusk", "Unknown"],     "source": VIDEO},
        {"kind": "cat", "col": "SURFACE_CONDITION_LABEL", "label": "Surface condition",
         "modalities": ["Dry", "Wet"],                            "source": VIDEO},
    ]),
    ("Geospatial", [
        {"kind": "cont", "col": "road_width_perp_m", "label": "Road width — perpendicular (m)", "source": GIS},
        {"kind": "cont", "col": "at_intersection",   "label": "At an intersection",              "source": GIS},
    ]),
    ("Temporal", [
        {"kind": "cont", "col": "hour",       "label": "Hour of day", "source": GPSTS},
        {"kind": "cont", "col": "is_weekend", "label": "Weekend",     "source": GPSTS},
        {"kind": "cat",  "col": "time_of_day", "label": "Time-of-day period",
         "modalities": ["Morning", "Afternoon", "Evening", "Night"], "source": GPSTS},
        {"kind": "cat",  "col": "season",     "label": "Season",
         "modalities": ["Spring", "Summer", "Autumn"],                "source": GPSTS},
    ]),
    ("Socio-demographic", [
        {"kind": "cat",  "col": "genre",          "label": "Gender",
         "modalities": ["male", "female"],                              "source": QUEST, "agg": "per_rider"},
        {"kind": "cont", "col": "age",            "label": "Age (years)",                "source": QUEST, "agg": "per_rider"},
        {"kind": "cat",  "col": "experience_bin", "label": "E-scooter experience (years)",
         "modalities": ["<0.5", "0.5-1", "1-2", ">2"],                  "source": QUEST, "agg": "per_rider"},
        {"kind": "cont", "col": "distance_km",    "label": "Trip distance (km)",          "source": GPS,   "agg": "per_rider"},
    ]),
]
print(f"{sum(len(rows) for _, rows in SECTIONS)} variables, {len(SECTIONS)} sections")


31 variables, 9 sections


## 6. Construction & écriture du `.tex`

In [7]:
HEADER = (
    r"\begin{longtable}{@{}p{3.8cm} p{1.8cm} c c c p{3.5cm}@{}}" "\n"
    r"\caption{Candidate variables for the discrete choice model (accelerate vs. decelerate/maintain)} \label{tab:variables_modele} \\" "\n"
    r"\toprule" "\n"
    r"\textbf{Variable} & \textbf{Modality} & \textbf{Min} & \textbf{Mean} & \textbf{Max} & \textbf{Source} \\" "\n"
    r"\midrule" "\n"
    r"\endfirsthead" "\n"
    r"\multicolumn{6}{c}{\tablename\ \thetable{} (continued)} \\" "\n"
    r"\toprule" "\n"
    r"\textbf{Variable} & \textbf{Modality} & \textbf{Min} & \textbf{Mean} & \textbf{Max} & \textbf{Source} \\" "\n"
    r"\midrule" "\n"
    r"\endhead" "\n"
    r"\midrule \multicolumn{6}{r}{\textit{continued \ldots}} \\" "\n"
    r"\endfoot" "\n"
    r"\bottomrule" "\n"
    r"\endlastfoot"
)
FOOTER = r"\end{longtable}"

section_blocks = []
for sec_title, rows in SECTIONS:
    block = render_section_header(sec_title)
    for row in rows:
        agg = row.get("agg")
        if row["kind"] == "cont":
            block.extend(render_cont_row(row["col"], row["label"], row["source"], agg))
        elif row["kind"] == "cat":
            block.extend(render_cat_block(row["col"], row["label"], row["modalities"], row["source"], agg))
    section_blocks.append(block)

joined = []
for i, block in enumerate(section_blocks):
    joined.extend(block)
    if i < len(section_blocks) - 1:
        joined.append(r"  \hline")

content = "\n".join([HEADER] + joined + [FOOTER]) + "\n"

Path(OUTPUT_PATH).write_text(content)
print(f"✔  {OUTPUT_PATH}")
print(f"   {len(content)} caractères, {len(content.splitlines())} lignes")


✔  /Volumes/My Passport/NEWMOB/variables_modele_2.tex
   4430 caractères, 86 lignes


## 7. Aperçu

In [8]:
# Aperçu (les 60 premières lignes)
print("\n".join(content.splitlines()[:60]))


\begin{longtable}{@{}p{3.8cm} p{1.8cm} c c c p{3.5cm}@{}}
\caption{Candidate variables for the discrete choice model (accelerate vs. decelerate/maintain)} \label{tab:variables_modele} \\
\toprule
\textbf{Variable} & \textbf{Modality} & \textbf{Min} & \textbf{Mean} & \textbf{Max} & \textbf{Source} \\
\midrule
\endfirsthead
\multicolumn{6}{c}{\tablename\ \thetable{} (continued)} \\
\toprule
\textbf{Variable} & \textbf{Modality} & \textbf{Min} & \textbf{Mean} & \textbf{Max} & \textbf{Source} \\
\midrule
\endhead
\midrule \multicolumn{6}{r}{\textit{continued \ldots}} \\
\endfoot
\bottomrule
\endlastfoot
\multicolumn{6}{l}{\textit{\small Kinematics}} \\
\addlinespace[1pt]
  Speed (km/h) & \textit{cont.} & 2.1 & 15.62 & 34.0 & GPS / IMU \\
  \hline
\multicolumn{6}{l}{\textit{\small VRU — Presence}} \\
\addlinespace[1pt]
  No. pedestrians in frame & \textit{cont.} & 0.0 & 3.72 & 17.0 & Video annotation \\
  No. cyclists in frame & \textit{cont.} & 0.0 & 0.31 & 8.0 & Video annotation \\
  No. 